In [11]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cmcsp10", con=connection2)

connection2.close()




In [12]:
base2

,csecod,csedes,cseedades,cseedahas,indsexadmcod,csetipcod,cseestregcod,cseanio,csectlapeflg,csectlgesflg
0,N1,(2008) CARTERA DE SERVICIOS NIÑO MENOR DE 1 AÑO,0,1,2,1,0,None,0,0
1,N2,(2008) CARTERA DE SERVICIOS DEL NIÑO DE 1 A ME...,1,2,2,1,0,None,1,0
2,N3,(2008) CARTERA DE SERVICIOS DEL NIÑO DE 2 A ME...,2,4,2,1,0,None,1,0
3,N4,(2008) CARTERA DE SERVICIOS DEL NIÑO DE 4 A ME...,4,5,2,1,0,None,1,0
4,N5,(2008) CARTERA DE SERVICIOS DEL NIÑO DE 5 A ME...,5,10,2,1,0,None,1,0
5,D1,(2008) CARTERA DE SERVICIOS DEL ADOLESCENTE DE...,10,15,2,1,0,None,0,0
6,D2,(2008) CARTERA DE SERVICIOS DEL ADOLESCENTE DE...,15,18,2,1,0,None,0,0
7,A1,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 18 A...,18,30,2,1,0,None,0,0
8,A2,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 30 A...,30,40,2,1,0,None,0,0
9,A3,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 40 A...,40,50,2,1,0,None,0,0


In [13]:
base2.columns

Index(['csecod', 'csedes', 'cseedades', 'cseedahas', 'indsexadmcod',
       'csetipcod', 'cseestregcod', 'cseanio', 'csectlapeflg', 'csectlgesflg'],
      dtype='object')

In [14]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmdia10 ()')
base2.to_sql(name='tmp_cmcsp10', con=engine3, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmaho10 FALSO CON LO OBTENIDO DEL ORACLE


query="""
UPDATE cmcsp10 
SET 
csedes =t.csedes,
cseedades =t.cseedades,
cseedahas =t.cseedahas,
csecod =t.csecod

FROM tmp_cmcsp10 t 
WHERE cmcsp10.csecod = t.csecod;

INSERT INTO cmcsp10 (csedes,cseedades,cseedahas,csecod) 

SELECT csedes,cseedades,cseedahas,csecod

FROM tmp_cmcsp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmcsp10 
    WHERE cmcsp10.csecod = tmp_cmcsp10.csecod
);


"""

c1= text(query)
connection3.execute(c1)




#BORRAMOS LAS TABLAS
base2 = pd.read_sql_query(f"SELECT * FROM cmcsp10", con=connection3)
query2="""
DROP TABLE tmp_cmcsp10;
"""

c2= text(query2)
connection3.execute(c2)
connection3.close()


In [15]:

#################################################################################################################################################################################################################################################################################

#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmcsp10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""
UPDATE dim_csep 
SET des_cse = t.csedes,
edad_des =t.cseedades,
edad_has =t.cseedahas

FROM tmp_cmcsp10 t 
WHERE dim_csep.cod_cse = t.csecod;

INSERT INTO dim_csep (cod_cse, des_cse, edad_des, edad_has) 
SELECT csecod,csedes,cseedades,cseedahas
FROM tmp_cmcsp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_csep 
    WHERE dim_csep.cod_cse = tmp_cmcsp10.csecod
);

DROP TABLE tmp_cmcsp10;
"""
c1= text(query)
connection4.execute(c1)
connection4.close()
